<a href="https://colab.research.google.com/github/Anh-field/Gas-EV/blob/main/Gas_and_Electric_Vehicles_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# Google Colab: install + enable sliders (run this cell first)
# =========================
!pip -q install ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

# =========================
# EV+HEV vs Gas Share Model (2011–2025) + Best-fit button (FIXED)
# Model: dE/dt = (r(t) - alpha) * E(t) * (1 - E(t))
# r(t) is a sigmoid ramp from r_min to r_max
# =========================

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# 1) DATA (Quarterly)
# -------------------------
rows = [
("2011-Q1",  79073, 2981068),
("2011-Q2",  60642, 3212541),
("2011-Q3",  62903, 3132967),
("2011-Q4",  83894, 3175861),
("2012-Q1", 113457, 3353882),
("2012-Q2", 121934, 3682774),
("2012-Q3", 118385, 3505565),
("2012-Q4", 134043, 3448237),
("2013-Q1", 136924, 3551651),
("2013-Q2", 159158, 3972720),
("2013-Q3", 159022, 3793322),
("2013-Q4", 134533, 3674235),
("2014-Q1", 124028, 3616896),
("2014-Q2", 163733, 4259211),
("2014-Q3", 157257, 4110708),
("2014-Q4", 126036, 3965679),
("2015-Q1", 107853, 3848907),
("2015-Q2", 135822, 4320844),
("2015-Q3", 133136, 4397097),
("2015-Q4", 121661, 4293615),
("2016-Q1", 101574, 3985500),
("2016-Q2", 125286, 4429811),
("2016-Q3", 141454, 4327679),
("2016-Q4", 138251, 4619473),
("2017-Q1", 123715, 3906979),
("2017-Q2", 141296, 4280350),
("2017-Q3", 154086, 4269005),
("2017-Q4", 147261, 4214695),
("2018-Q1", 127984, 3963880),
("2018-Q2", 157309, 4326446),
("2018-Q3", 180320, 4096995),
("2018-Q4", 203381, 4157231),
("2019-Q1", 148370, 3835111),
("2019-Q2", 199893, 4213815),
("2019-Q3", 194700, 4111632),
("2019-Q4", 176094, 4073297),
("2020-Q1", 153538, 3323546),
("2020-Q2", 127223, 2803357),
("2020-Q3", 216065, 3679747),
("2020-Q4", 265830, 3897726),
("2021-Q1", 299854, 3597360),
("2021-Q2", 390183, 4000758),
("2021-Q3", 364198, 3025201),
("2021-Q4", 380402, 2885017),
("2022-Q1", 392496, 2902522),
("2022-Q2", 428902, 3059413),
("2022-Q3", 409101, 2995984),
("2022-Q4", 467316, 3118605),
("2023-Q1", 521636, 2949031),
("2023-Q2", 668275, 3422100),
("2023-Q3", 710516, 3258323),
("2023-Q4", 733245, 3159353),
("2024-Q1", 659933, 3071722),
("2024-Q2", 777137, 3280422),
("2024-Q3", 819828, 3059894),
("2024-Q4", 914293, 3285189),
("2025-Q1", 845025, 3065751),
("2025-Q2", 900470, 3280149),
("2025-Q3", 981215, 3116275),
("2025-Q4", 818739, 3206279),
]

def quarter_to_year_float(yq: str) -> float:
    y, q = yq.split("-Q")
    y = int(y); q = int(q)
    return y + (q - 1) * 0.25  # Q1=0.00, Q2=0.25, Q3=0.50, Q4=0.75

T_year = np.array([quarter_to_year_float(r[0]) for r in rows], dtype=float)
BEV = np.array([r[1] for r in rows], dtype=float)
GAS = np.array([r[2] for r in rows], dtype=float)

TOTAL = BEV + GAS
E_share = BEV / TOTAL
G_share = 1.0 - E_share  # enforce E+G=1

# -------------------------
# 2) MODEL
# -------------------------
def r_of_t(t, rmin, rmax, k, tmid):
    # sigmoid ramp
    return rmin + (rmax - rmin) / (1 + np.exp(-k * (t - tmid)))

def dE_dt(t, E, rmin, rmax, k, tmid, alpha):
    rt = r_of_t(t, rmin, rmax, k, tmid)
    return (rt - alpha) * E * (1 - E)

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

# Fast simulator for fitting (Euler; faster than solve_ivp for many trials)
def simulate_euler(t_data, E0, rmin, rmax, k, tmid, alpha, dt=0.01):
    t_end = float(t_data.max())
    n = int(np.ceil(t_end / dt))
    E = float(E0)
    E = min(max(E, 1e-6), 1 - 1e-6)

    ts = np.linspace(0.0, n * dt, n + 1)
    Es = np.empty(n + 1, dtype=float)
    Es[0] = E

    for i in range(n):
        t = ts[i]
        rt = r_of_t(t, rmin, rmax, k, tmid)
        E = E + dt * (rt - alpha) * E * (1 - E)
        # keep inside (0,1)
        if E <= 1e-10: E = 1e-10
        if E >= 1 - 1e-10: E = 1 - 1e-10
        Es[i + 1] = E

    return np.interp(t_data, ts, Es)

# -------------------------
# 3) UI CONTROLS
# -------------------------
start = widgets.FloatSlider(
    value=2011.00, min=float(T_year[0]), max=float(T_year[-1] - 1),
    step=0.25, description="Start", readout_format=".2f"
)

# Defaults tuned for 2011–2025
rmin  = widgets.FloatSlider(value=0.04, min=-0.50, max=1.50, step=0.01, description="r_min")
rmax  = widgets.FloatSlider(value=0.65, min=-0.50, max=2.50, step=0.01, description="r_max")
k_sl  = widgets.FloatSlider(value=1.20, min=0.00,  max=10.0, step=0.10, description="k")
tmid  = widgets.FloatSlider(value=9.00, min=0.00,  max=20.0, step=0.25, description="t_mid")
alpha = widgets.FloatSlider(value=0.06, min=-0.50, max=1.50, step=0.01, description="alpha")

show_phase = widgets.Checkbox(value=True, description="Show phase portrait")
show_r     = widgets.Checkbox(value=True, description="Show r(t)")

reset_btn = widgets.Button(description="Reset (2011–2025 defaults)", button_style="info")
fit_btn   = widgets.Button(description="Best-fit (random search)", button_style="warning")

trials = widgets.IntSlider(value=400, min=50, max=2000, step=50, description="trials")
dt_fit = widgets.FloatSlider(value=0.01, min=0.005, max=0.05, step=0.005, description="dt (fit)")

out = widgets.Output()

def reset(_):
    start.value = 2011.00
    rmin.value = 0.04
    rmax.value = 0.65
    k_sl.value = 1.20
    tmid.value = 9.00
    alpha.value = 0.06
    show_phase.value = True
    show_r.value = True

reset_btn.on_click(reset)

# -------------------------
# 4) Plotting function (smooth solve_ivp)
# -------------------------
def draw_plots(sy, rmin_v, rmax_v, k_v, tmid_v, alpha_v):
    mask = T_year >= sy
    T = T_year[mask]
    Edata = E_share[mask]
    Gdata = 1 - Edata

    t0 = float(T[0])
    t_data = T - t0
    t_span = (float(t_data.min()), float(t_data.max()))
    t_eval = np.linspace(t_span[0], t_span[1], 2000)

    E0 = float(Edata[0])
    E0 = min(max(E0, 1e-6), 1 - 1e-6)

    sol = solve_ivp(
        fun=lambda tt, yy: dE_dt(tt, yy, rmin_v, rmax_v, k_v, tmid_v, alpha_v),
        t_span=t_span, y0=[E0], t_eval=t_eval,
        rtol=1e-7, atol=1e-9
    )

    Tm = sol.t + t0
    Em = sol.y[0]
    Gm = 1 - Em

    # score on observed quarters
    E_pred = np.interp(T, Tm, Em)
    scoreE = rmse(Edata, E_pred)

    # Plot shares vs time
    plt.figure(figsize=(10, 6))
    plt.plot(Tm, Em, linewidth=2, label="EV+HEV share (model)")
    plt.plot(Tm, Gm, linewidth=2, label="Gas share (model)")
    plt.scatter(T, Edata, s=25, label="EV+HEV share (data)")
    plt.scatter(T, Gdata, s=25, label="Gas share (data)")
    plt.title("EV+HEV vs Gas — dE/dt=(r(t)-alpha)E(1-E)")
    plt.xlabel("Year")
    plt.ylabel("Market share (E+G=1)")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.show()

    # r(t) plot (FIXED: no extra alpha argument)
    if show_r.value:
        t_model = sol.t
        r_vals = r_of_t(t_model, rmin_v, rmax_v, k_v, tmid_v)
        plt.figure(figsize=(10, 3.5))
        plt.plot(Tm, r_vals, linewidth=2)
        plt.axhline(alpha_v, linestyle="--")
        plt.title("r(t) (solid) and alpha (dashed)")
        plt.xlabel("Year")
        plt.ylabel("Rate per year")
        plt.grid(True, alpha=0.35)
        plt.show()

    # phase portrait
    if show_phase.value:
        plt.figure(figsize=(6.5, 6))
        plt.plot(Em, Gm, linewidth=2, label="Trajectory (model)")
        plt.scatter(Edata, Gdata, s=25, label="Data (E,G)")
        plt.title("Phase Portrait (E vs G)")
        plt.xlabel("E = EV+HEV share")
        plt.ylabel("G = Gas share (1-E)")
        plt.grid(True, alpha=0.35)
        plt.legend()
        plt.show()

    print(f"Data window: {T[0]:.2f} to {T[-1]:.2f} (quarter step = 0.25 years)")
    print(f"RMSE(E share) = {scoreE:.5f}")
    print("Parameters:")
    print(f" r_min={rmin_v:.3f}, r_max={rmax_v:.3f}, k={k_v:.2f}, t_mid={tmid_v:.2f}, alpha={alpha_v:.3f}")
    print("Interpretation tip: growth mainly when r(t) > alpha.")

def update(*_):
    with out:
        clear_output(wait=True)
        draw_plots(start.value, rmin.value, rmax.value, k_sl.value, tmid.value, alpha.value)

for w in [start, rmin, rmax, k_sl, tmid, alpha, show_phase, show_r]:
    w.observe(update, names="value")

# -------------------------
# 5) Best-fit button (random search using Euler simulation)
# -------------------------
def best_fit(_):
    with out:
        clear_output(wait=True)
        print("Running random search… (fast Euler simulation)\n")

        sy = start.value
        mask = T_year >= sy
        T = T_year[mask]
        Edata = E_share[mask]

        t0 = float(T[0])
        t_data = T - t0

        E0 = float(Edata[0])
        E0 = min(max(E0, 1e-6), 1 - 1e-6)

        rng = np.random.default_rng(0)

        # Search ranges (tuned for 2011–2025)
        rmin_lo, rmin_hi   = -0.05, 0.20
        rmax_lo, rmax_hi   =  0.20, 1.50
        k_lo, k_hi         =  0.30, 4.00
        tmid_lo, tmid_hi   =  5.00, 12.00   # years after start
        alpha_lo, alpha_hi =  0.00, 0.20

        best_params = None
        best_score = float("inf")

        N = int(trials.value)
        dt = float(dt_fit.value)

        # include current slider setting
        candidates = [(rmin.value, rmax.value, k_sl.value, tmid.value, alpha.value)]

        # random samples
        for _i in range(N):
            rrmin  = rng.uniform(rmin_lo, rmin_hi)
            rrmax  = rng.uniform(rmax_lo, rmax_hi)
            kk     = rng.uniform(k_lo, k_hi)
            ttmid  = rng.uniform(tmid_lo, tmid_hi)
            aalpha = rng.uniform(alpha_lo, alpha_hi)
            if rrmax <= rrmin:
                continue
            candidates.append((rrmin, rrmax, kk, ttmid, aalpha))

        # evaluate
        for rrmin, rrmax, kk, ttmid, aalpha in candidates:
            E_pred = simulate_euler(t_data, E0, rrmin, rrmax, kk, ttmid, aalpha, dt=dt)
            score = rmse(Edata, E_pred)
            if score < best_score:
                best_score = score
                best_params = (rrmin, rrmax, kk, ttmid, aalpha)

        if best_params is None:
            print("No valid parameters found. Increase trials or widen ranges.")
            return

        rrmin, rrmax, kk, ttmid, aalpha = best_params

        print(f"Best (Euler-fit) RMSE(E share) ≈ {best_score:.5f}")
        print("Updating sliders to best-fit values and re-plotting with solve_ivp...\n")

        # Update sliders (this will also trigger update)
        rmin.value  = float(rrmin)
        rmax.value  = float(rrmax)
        k_sl.value  = float(kk)
        tmid.value  = float(ttmid)
        alpha.value = float(aalpha)

fit_btn.on_click(best_fit)

# -------------------------
# 6) Display UI
# -------------------------
controls_left  = widgets.VBox([widgets.Label("Window"), start, reset_btn])
controls_mid   = widgets.VBox([widgets.Label("r(t) controls"), rmin, rmax, k_sl, tmid])
controls_right = widgets.VBox([widgets.Label("Drag / options"), alpha, show_r, show_phase])
controls_fit   = widgets.VBox([widgets.Label("Auto-fit"), trials, dt_fit, fit_btn])

display(widgets.HBox([controls_left, controls_mid, controls_right, controls_fit]), out)
update()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.0 MB/s eta 0:00:00


Output()